# TabPFN vs XGBoost: 튜닝 없는 AI 모델이 튜닝한 XGBoost를 이길까?

데이터팝콘 테크블로그 글 「TabPFN vs XGBoost」의 실습 노트북입니다.

- 비교 모델: XGBoost 기본값 / XGBoost + Optuna 30회 튜닝 / TabPFN-2 (튜닝 없음)
- 실험 ①: 작은 공개 데이터 6종, 70:30 랜덤 분할 5회
- 실험 ②: 학습 데이터를 100 → 300 → 1,000 → 3,000행으로 늘리며 비교 (미국 소득, 캘리포니아 집값)

`런타임 → 런타임 유형 변경 → T4 GPU`를 켜면 TabPFN이 훨씬 빨라집니다. CPU에서도 돌아갑니다.

In [ ]:
# 1) 설치: Colab에는 xgboost·scikit-learn이 이미 있습니다. TabPFN과 Optuna만 추가합니다.
!pip install -q "tabpfn==10.2.0" optuna

In [ ]:
# 2) 불러오기와 설정
import os, time, warnings
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"   # CPU에서 1,000행 넘는 데이터도 허용
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, optuna, xgboost as xgb
from sklearn.datasets import load_breast_cancer, load_diabetes, fetch_openml, fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import roc_auc_score, mean_squared_error
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_TRIALS = 30          # XGBoost 튜닝 횟수
N_SPLITS_A = 5         # 실험 ① 반복 횟수
N_SEEDS_B = 3          # 실험 ② 반복 횟수
SIZES_B = [100, 300, 1000, 3000]
print("TabPFN 실행 장치:", DEVICE)

In [ ]:
# 3) 데이터 불러오기: sklearn 내장 데이터와 OpenML 공개 데이터
OPENML_IDS = {"credit_g": 31, "pima_diabetes": 37, "blood_transfusion": 1464, "adult": 1590, "auto_mpg": 196}

def load(name):
    if name == "breast_cancer":
        X, y = load_breast_cancer(return_X_y=True, as_frame=True); return X, y, "cls"
    if name == "diabetes_reg":
        X, y = load_diabetes(return_X_y=True, as_frame=True); return X, y, "reg"
    if name == "california":
        X, y = fetch_california_housing(return_X_y=True, as_frame=True); return X, y, "reg"
    try:
        d = fetch_openml(data_id=OPENML_IDS[name], as_frame=True, parser="auto")
    except TypeError:  # 일부 데이터는 pandas 파서가 실패해 liac-arff로 다시 읽습니다
        d = fetch_openml(data_id=OPENML_IDS[name], as_frame=True, parser="liac-arff")
    X, y = d.data, d.target
    task = "reg" if name == "auto_mpg" else "cls"
    y = pd.Series(pd.factorize(y, sort=True)[0], index=y.index) if task == "cls" else y.astype(float)
    return X, y, task

def prep(X):
    # XGBoost용: 문자열 열은 category 타입 / TabPFN용: 같은 열을 정수 코드로 바꾸고 위치를 알려 줌
    X = X.copy()
    cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    for c in cat_cols:
        X[c] = X[c].astype(str).astype("category")
    Xn = X.copy()
    for c in cat_cols:
        Xn[c] = Xn[c].cat.codes.astype(float)
    return Xn.to_numpy(dtype=float), [X.columns.get_loc(c) for c in cat_cols], X

X, y, task = load("credit_g")
print(X.shape, task); X.head()

In [ ]:
# 4) 세 모델을 같은 분할로 학습·평가하는 함수
def xgb_model(task, **p):
    base = dict(tree_method="hist", enable_categorical=True, verbosity=0); base.update(p)
    return xgb.XGBClassifier(eval_metric="logloss", **base) if task == "cls" else xgb.XGBRegressor(**base)

def evaluate(task, y_true, pred):
    # 분류는 AUC(높을수록 좋음), 회귀는 RMSE(낮을수록 좋음)
    return roc_auc_score(y_true, pred) if task == "cls" else float(np.sqrt(mean_squared_error(y_true, pred)))

def predict(m, task, X):
    return m.predict_proba(X)[:, 1] if task == "cls" else m.predict(X)

def tune_xgb(task, X, y, seed):
    cv = StratifiedKFold(3, shuffle=True, random_state=seed) if task == "cls" else KFold(3, shuffle=True, random_state=seed)
    scoring = "roc_auc" if task == "cls" else "neg_root_mean_squared_error"
    def objective(tr):
        p = dict(n_estimators=tr.suggest_int("n_estimators", 50, 800, log=True),
                 learning_rate=tr.suggest_float("learning_rate", 0.005, 0.3, log=True),
                 max_depth=tr.suggest_int("max_depth", 2, 10),
                 subsample=tr.suggest_float("subsample", 0.5, 1.0),
                 colsample_bytree=tr.suggest_float("colsample_bytree", 0.4, 1.0),
                 min_child_weight=tr.suggest_float("min_child_weight", 0.5, 20, log=True),
                 reg_lambda=tr.suggest_float("reg_lambda", 1e-3, 10, log=True))
        return cross_val_score(xgb_model(task, random_state=seed, **p), X, y, cv=cv, scoring=scoring).mean()
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return study.best_params

def run_three(task, Xf_tr, Xf_te, Xn_tr, Xn_te, cat_idx, y_tr, y_te, seed):
    out = []
    t = time.time(); m = xgb_model(task, random_state=seed).fit(Xf_tr, y_tr)
    s = evaluate(task, y_te, predict(m, task, Xf_te)); out.append(("XGBoost 기본값", s, time.time() - t))

    t = time.time(); best = tune_xgb(task, Xf_tr, y_tr, seed)
    m = xgb_model(task, random_state=seed, **best).fit(Xf_tr, y_tr)
    s = evaluate(task, y_te, predict(m, task, Xf_te)); out.append((f"XGBoost 튜닝({N_TRIALS}회)", s, time.time() - t))

    t = time.time()
    Model = TabPFNClassifier if task == "cls" else TabPFNRegressor
    m = Model.create_default_for_version(ModelVersion.V2, device=DEVICE, random_state=seed,
                                         categorical_features_indices=cat_idx or None,
                                         ignore_pretraining_limits=True)
    m.fit(Xn_tr, y_tr.to_numpy())
    s = evaluate(task, y_te, predict(m, task, Xn_te)); out.append(("TabPFN-2 (튜닝 없음)", s, time.time() - t))
    return out

In [ ]:
# 5) 실험 ①: 작은 데이터 6종 × 랜덤 분할 5회
rows = []
for name in ["breast_cancer", "credit_g", "pima_diabetes", "blood_transfusion", "diabetes_reg", "auto_mpg"]:
    X, y, task = load(name); Xn, cat_idx, Xf = prep(X)
    for seed in range(N_SPLITS_A):
        tr, te = train_test_split(np.arange(len(y)), test_size=0.3, random_state=seed,
                                  stratify=y if task == "cls" else None)
        for model, score, sec in run_three(task, Xf.iloc[tr], Xf.iloc[te], Xn[tr], Xn[te], cat_idx,
                                           y.iloc[tr], y.iloc[te], seed):
            rows.append(dict(dataset=name, task=task, seed=seed, model=model, score=score, sec=sec))
    print("완료:", name)
A = pd.DataFrame(rows)
A.pivot_table(index="dataset", columns="model", values="score").round(3)

In [ ]:
# 분할마다 1등 모델 세기 (분류는 AUC 최대, 회귀는 RMSE 최소)
def winner(g):
    return g.loc[g.score.idxmax() if g.task.iloc[0] == "cls" else g.score.idxmin(), "model"]
A.groupby(["dataset", "seed"]).apply(winner).value_counts()

In [ ]:
# 6) 실험 ②: 학습 데이터 양을 늘려 가며 비교 (테스트 2,000행 고정)
rows = []
for name in ["adult", "california"]:
    X, y, task = load(name); Xn, cat_idx, Xf = prep(X)
    for seed in range(N_SEEDS_B):
        perm = np.random.RandomState(100 + seed).permutation(len(y))
        te, pool = perm[:2000], perm[2000:]
        for n in SIZES_B:
            tr = pool[:n]
            for model, score, sec in run_three(task, Xf.iloc[tr], Xf.iloc[te], Xn[tr], Xn[te], cat_idx,
                                               y.iloc[tr], y.iloc[te], seed):
                rows.append(dict(dataset=name, task=task, n_train=n, seed=seed, model=model, score=score, sec=sec))
        print("완료:", name, "seed", seed)
B = pd.DataFrame(rows)
B.pivot_table(index=["dataset", "n_train"], columns="model", values="score").round(4)

In [ ]:
# 7) 학습 곡선 그리기 (차트 글자는 Colab 기본 폰트에서 깨지지 않게 영어로 씁니다)
import matplotlib.pyplot as plt
label = {"XGBoost 기본값": "XGBoost default", f"XGBoost 튜닝({N_TRIALS}회)": f"XGBoost tuned ({N_TRIALS})",
         "TabPFN-2 (튜닝 없음)": "TabPFN-2 (no tuning)"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, metric) in zip(axes, [("adult", "AUC (higher is better)"), ("california", "RMSE (lower is better)")]):
    g = B[B.dataset == name].groupby(["model", "n_train"]).score.mean().unstack(0)
    for col in g.columns:
        ax.plot(g.index, g[col], marker="o", label=label[col], linewidth=2.5 if "TabPFN" in col else 1.5)
    ax.set_xscale("log"); ax.minorticks_off(); ax.set_xticks(SIZES_B); ax.set_xticklabels(SIZES_B)
    ax.set_title(name); ax.set_xlabel("training rows"); ax.set_ylabel(metric); ax.grid(alpha=.3)
axes[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
# 8) 걸린 시간 비교 (초, 학습+예측. 튜닝 모델은 튜닝 시간 포함)
pd.concat([A.assign(n_train="small"), B]).groupby(["n_train", "model"]).sec.mean().unstack().round(1)

## 다음으로 해 볼 것

- 내 데이터로 바꾸려면 `load()`에 CSV를 읽는 분기를 하나 추가하면 됩니다. 분류 타깃은 0/1, 회귀 타깃은 숫자여야 합니다.
- 최신 TabPFN-3.5는 [Prior Labs](https://ux.priorlabs.ai)에서 라이선스에 동의하고 API 키를 받아 `TABPFN_TOKEN`으로 넣으면 `ModelVersion.V3_5`로 바꿔 돌릴 수 있습니다. 2.5 이후 가중치는 비상업 라이선스입니다.